<a href="https://colab.research.google.com/github/emankhalid019/sql-from-scratch/blob/main/Day02_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Theory + all commands
# Aggregate Functions, Like/Wildcards, In,Between, Aliases, Joins, Union



import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cur = conn.cursor()

def run(query):
    """Executes a query. Returns a table if it's a SELECT, else a status message."""
    cur.execute(query)
    conn.commit()
    if query.strip().upper().startswith("SELECT"):
        cols = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=cols)
    return "Query executed successfully."




#  sample tables used throughout

run("""
CREATE TABLE Employees (
    EmployeeID INTEGER,
    Name TEXT,
    Age INTEGER,
    Salary INTEGER,
    DepartmentID INTEGER
);
""")

run("""
CREATE TABLE Departments (
    DepartmentID INTEGER,
    DepartmentName TEXT
);
""")

run("INSERT INTO Employees VALUES (1, 'Ali', 25, 50000, 1);")
run("INSERT INTO Employees VALUES (2, 'Sara', 30, 60000, 2);")
run("INSERT INTO Employees VALUES (3, 'Ahmed', 28, 55000, 1);")
run("INSERT INTO Employees VALUES (4, 'Ayesha', 35, 70000, 3);")
run("INSERT INTO Employees VALUES (5, 'Zarmeen', 22, 45000, NULL);")  # no department yet

run("INSERT INTO Departments VALUES (1, 'IT');")
run("INSERT INTO Departments VALUES (2, 'HR');")
run("INSERT INTO Departments VALUES (3, 'Finance');")
run("INSERT INTO Departments VALUES (4, 'Marketing');")  # no employees yet

print(run("SELECT * FROM Employees;"))
print(run("SELECT * FROM Departments;"))







## SQL MIN()
# MIN() returns the smallest value in a column.
print(run("SELECT MIN(Salary) AS LowestSalary FROM Employees;"))






## SQL MAX()
# MAX() returns the largest value in a column.
print(run("SELECT MAX(Salary) AS HighestSalary FROM Employees;"))





## SQL COUNT()
# COUNT() returns the number of rows that match a condition.
print(run("SELECT COUNT(*) AS TotalEmployees FROM Employees;"))
print(run("SELECT COUNT(*) AS ITEmployees FROM Employees WHERE DepartmentID = 1;"))
# COUNT(column) counts only non-NULL values in that column
print(run("SELECT COUNT(DepartmentID) AS AssignedEmployees FROM Employees;"))




## SQL SUM()
# SUM() adds up all numeric values in a column
print(run("SELECT SUM(Salary) AS TotalSalary FROM Employees;"))





## SQL AVG()
# AVG() returns the average value of a column.
# Note: Aggregate functions ignore NULL values automatically.
print(run("SELECT AVG(Salary) AS AverageSalary FROM Employees;"))
print(run("SELECT AVG(Age) AS AverageAge FROM Employees;"))





## SQL LIKE
# LIKE is used inside WHERE to search for a specified pattern in a column.
print(run("SELECT * FROM Employees WHERE Name LIKE 'A%';"))   # starts with A
print(run("SELECT * FROM Employees WHERE Name LIKE '%a';"))   # ends with a
print(run("SELECT * FROM Employees WHERE Name LIKE '%ee%';")) # contains 'ee'
print(run("SELECT * FROM Employees WHERE Name LIKE '_ara';")) # 2nd-5th letters = 'ara'




## SQL WILDCARDS
# Wildcards are used with LIKE to represent unknown character(s).
# %   -> represents zero, one, or multiple characters
# _   -> represents exactly one character
# []  -> (SQL Server only) any single character within brackets, e.g. [ab]
# In SQLite, only % and _ are supported

print(run("SELECT * FROM Employees WHERE Name LIKE 'A_i';"))   # A, any 1 char, i -> "Ali"




## SQL IN
# IN allows matching a column against multiple possible values.
print(run("SELECT * FROM Employees WHERE DepartmentID IN (1, 2);"))
print(run("SELECT * FROM Employees WHERE Name IN ('Ali', 'Sara', 'Ahmed');"))

# IN is a shorthand for multiple OR conditions:
print(run("SELECT * FROM Employees WHERE DepartmentID = 1 OR DepartmentID = 2;"))




## SQL BETWEEN
# BETWEEN selects values within a given range (inclusive of both ends).
print(run("SELECT * FROM Employees WHERE Salary BETWEEN 50000 AND 65000;"))
print(run("SELECT * FROM Employees WHERE Age BETWEEN 25 AND 30;"))

# NOT BETWEEN -> excludes the range
print(run("SELECT * FROM Employees WHERE Salary NOT BETWEEN 50000 AND 65000;"))





## SQL ALIASES
# Aliases give a table or column a temporary, more readable name
# Column alias syntax: column_name AS alias_name
# Table alias syntax:  table_name AS alias_name

print(run("SELECT Name AS EmployeeName, Salary AS MonthlySalary FROM Employees;"))

# Table alias (useful in joins, shown below)
print(run("SELECT e.Name, e.Salary FROM Employees AS e;"))

# AS is optional in most databases:
print(run("SELECT Name EmployeeName FROM Employees;"))



## SQL JOINS - overview
# A JOIN combines rows from two or more tables based on a related column
# Types of joins:
#   INNER JOIN  -> only matching rows in both tables
#   LEFT JOIN   -> all rows from left table + matches from right (NULL if no match)
#   RIGHT JOIN  -> all rows from right table + matches from left (NULL if no match)
#   FULL JOIN   -> all rows from both tables (matched where possible)
#   SELF JOIN   -> a table joined with itself





## SQL INNER JOIN
# Returns only rows where there is a match in BOTH tables.
print(run("""
SELECT Employees.Name, Departments.DepartmentName
FROM Employees
INNER JOIN Departments
ON Employees.DepartmentID = Departments.DepartmentID;
"""))
# Zarmeen (no department) and Marketing (no employees) are excluded.




## SQL LEFT JOIN  (LEFT OUTER JOIN)
# Returns ALL rows from the left table, plus matching rows from the
# right table. If there's no match, right table columns show NULL
print(run("""
SELECT Employees.Name, Departments.DepartmentName
FROM Employees
LEFT JOIN Departments
ON Employees.DepartmentID = Departments.DepartmentID;
"""))
# Zarmeen appears with DepartmentName = NULL since she has no department




## SQL RIGHT JOIN  (RIGHT OUTER JOIN)
# Returns ALL rows from the right table, plus matching rows from the
# left table. If there's no match, left table columns show NULL.
# Note: SQLite does not support RIGHT JOIN directly, so it's simulated
# here by swapping table order and using LEFT JOIN.
print(run("""
SELECT Employees.Name, Departments.DepartmentName
FROM Departments
LEFT JOIN Employees
ON Employees.DepartmentID = Departments.DepartmentID;
"""))
# Marketing appears with Name = NULL since no employee belongs to it.

# Equivalent standard syntax on databases that support RIGHT JOIN
#   SELECT Employees.Name, Departments.DepartmentName
#   FROM Employees
#   RIGHT JOIN Departments
#   ON Employees.DepartmentID = Departments.DepartmentID;




## SQL FULL JOIN  (FULL OUTER JOIN)
# Returns ALL rows when there is a match in EITHER table.
# Rows with no match on either side show NULL for the missing side.
# Note: SQLite doesn't support FULL JOIN directly either, so it's
# simulated below with LEFT JOIN + UNION + RIGHT-style LEFT JOIN.
print(run("""
SELECT Employees.Name, Departments.DepartmentName
FROM Employees
LEFT JOIN Departments
ON Employees.DepartmentID = Departments.DepartmentID

UNION

SELECT Employees.Name, Departments.DepartmentName
FROM Departments
LEFT JOIN Employees
ON Employees.DepartmentID = Departments.DepartmentID;
"""))
# This shows every employee AND every department, matched where possible.
# Equivalent standard syntax on databases that support FULL JOIN
#   SELECT Employees.Name, Departments.DepartmentName
#   FROM Employees
#   FULL JOIN Departments
#   ON Employees.DepartmentID = Departments.DepartmentID;



## SQL SELF JOIN
# A self join joins a table with itself, useful for comparing rows
# within the same table (e.g. finding employees in the same department)

print(run("""
SELECT A.Name AS Employee1, B.Name AS Employee2, A.DepartmentID
FROM Employees A, Employees B
WHERE A.DepartmentID = B.DepartmentID
AND A.EmployeeID <> B.EmployeeID;
"""))
# Shows pairs of employees who work in the same department
# (Ali and Ahmed are both in DepartmentID 1)




## SQL UNIO
# UNION combines the result of two or more SELECT statements into one
# result set. Each SELECT must have the same number of columns, in the
# same order, with compatible data types.
# UNION removes duplicate rows by default. UNION ALL keeps duplicates.

run("""
CREATE TABLE FormerEmployees (
    Name TEXT
);
""")
run("INSERT INTO FormerEmployees VALUES ('Bilal');")
run("INSERT INTO FormerEmployees VALUES ('Ali');")  # duplicate name, for demo

# UNION (combines current + former employee names, removes duplicates)
print(run("""
SELECT Name FROM Employees
UNION
SELECT Name FROM FormerEmployees;
"""))

# UNION ALL (combines and KEEPS duplicates)
print(run("""
SELECT Name FROM Employees
UNION ALL
SELECT Name FROM FormerEmployees;
"""))

conn.close()




# Keyword         Purpose                                Example

# MIN()           Smallest value in a column              MIN(Salary)
# MAX()           Largest value in a column               MAX(Salary)
# COUNT()         Number of rows                          COUNT(*)
# SUM()           Total of numeric values                 SUM(Salary)
# AVG()           Average value                           AVG(Salary)
# LIKE            Pattern matching in WHERE                WHERE Name LIKE 'A%'
# Wildcards       % = any chars, _ = one char              WHERE Name LIKE 'A_i'
# IN              Match one of several values              WHERE DepartmentID IN (1,2)
# BETWEEN         Value within a range (inclusive)          WHERE Salary BETWEEN 50000 AND 65000
# Aliases         Temporary name for column/table           SELECT Name AS EmployeeName
# INNER JOIN      Only matching rows in both tables          FROM A INNER JOIN B ON A.id=B.id
# LEFT JOIN       All left rows + matches from right         FROM A LEFT JOIN B ON A.id=B.id
# RIGHT JOIN      All right rows + matches from left         FROM A RIGHT JOIN B ON A.id=B.id
# FULL JOIN       All rows from both tables                  FROM A FULL JOIN B ON A.id=B.id
# SELF JOIN       Table joined with itself                   FROM A a, A b WHERE a.id <> b.id
# UNION           Combine SELECT results, removes duplicates  SELECT... UNION SELECT...
# UNION ALL       Combine SELECT results, keeps duplicates    SELECT... UNION ALL SELECT...


   EmployeeID     Name  Age  Salary  DepartmentID
0           1      Ali   25   50000           1.0
1           2     Sara   30   60000           2.0
2           3    Ahmed   28   55000           1.0
3           4   Ayesha   35   70000           3.0
4           5  Zarmeen   22   45000           NaN
   DepartmentID DepartmentName
0             1             IT
1             2             HR
2             3        Finance
3             4      Marketing
   LowestSalary
0         45000
   HighestSalary
0          70000
   TotalEmployees
0               5
   ITEmployees
0            2
   AssignedEmployees
0                  4
   TotalSalary
0       280000
   AverageSalary
0        56000.0
   AverageAge
0        28.0
   EmployeeID    Name  Age  Salary  DepartmentID
0           1     Ali   25   50000             1
1           3   Ahmed   28   55000             1
2           4  Ayesha   35   70000             3
   EmployeeID    Name  Age  Salary  DepartmentID
0           2    Sara   30   60000